In [12]:
import pandas as pd

# Load the CSV files
jobreq_df = pd.read_csv('all_jobreq_extractedentity.csv')
resumes_df = pd.read_csv('all_resumes_extractedentity.csv')

# Ensure entities are in the correct format (convert string to list if necessary)
jobreq_df['entities'] = jobreq_df['entities'].apply(lambda x: x.split(', ') if isinstance(x, str) else x)
resumes_df['entities'] = resumes_df['entities'].apply(lambda x: x.split(', ') if isinstance(x, str) else x)

# Function to calculate Jaccard Similarity
def jaccard_similarity(set1, set2):
    intersection = len(set(set1).intersection(set2))
    union = len(set(set1).union(set2))
    return intersection / union if union != 0 else 0

# Create a DataFrame to store the results
results = []

# Iterate over each resume and job description
for resume_index, resume_row in resumes_df.iterrows():
    resume_entities = resume_row['entities']
    for job_index, job_row in jobreq_df.iterrows():
        job_entities = job_row['entities']
        job_title = job_row['job_title']
        
        # Calculate Jaccard Similarity
        similarity = jaccard_similarity(resume_entities, job_entities)
        
        # Append the result
        results.append({
            'resume_id': resume_index,
            'job_id': job_index,
            'job_title': job_title,
            'jaccard_similarity': similarity
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Save the results to a CSV file (optional)
results_df.to_csv('jaccard_similarity_results.csv', index=False)

# Display the results
print(results_df.head())

   resume_id  job_id              job_title  jaccard_similarity
0          0       0        ACCOUNTING HEAD            0.000000
1          0       1  ACCOUNTING SUPERVISOR            0.000000
2          0       2    ACCOUNTS SPECIALIST            0.000000
3          0       3        AREA SUPERVISOR            0.000000
4          0       4        AREA SUPERVISOR            0.020619


In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Load the Jaccard Similarity results
results_df = pd.read_csv('jaccard_similarity_results.csv')

# Add a 'job_id' column (assuming each job description has a unique ID)
results_df['job_id'] = results_df.groupby('job_title').ngroup()  # Assign unique IDs to job titles

# Features and target
X = results_df[['jaccard_similarity']]  # Feature
y = results_df['job_id']  # Target (multi-class)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Address Imbalanced Data using SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Initialize and train the XGBoost model
model = xgb.XGBClassifier(
    objective='multi:softprob',  # Multi-class classification with probabilities
    eval_metric='mlogloss',      # Logarithmic loss for multi-class evaluation
    random_state=42,
    n_estimators=500,           # Increase number of trees
    early_stopping_rounds=20,   # Early stopping to prevent overfitting
    num_class=len(y.unique()),  # Number of unique job classes
    max_depth=6,                # Experiment with different depths
    learning_rate=0.1,          # Experiment with different learning rates
)

# Fit the model
model.fit(X_train_res, y_train_res, eval_set=[(X_test, y_test)], verbose=True)

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)  # Probability distribution over all classes


c:\Users\Acer\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


[0]	validation_0-mlogloss:4.58239
[1]	validation_0-mlogloss:4.56405
[2]	validation_0-mlogloss:4.54834
[3]	validation_0-mlogloss:4.53473
[4]	validation_0-mlogloss:4.52276
[5]	validation_0-mlogloss:4.51212
[6]	validation_0-mlogloss:4.50259
[7]	validation_0-mlogloss:4.49421
[8]	validation_0-mlogloss:4.48674
[9]	validation_0-mlogloss:4.48011
[10]	validation_0-mlogloss:4.47411
[11]	validation_0-mlogloss:4.46888
[12]	validation_0-mlogloss:4.46406
[13]	validation_0-mlogloss:4.45976
[14]	validation_0-mlogloss:4.45596
[15]	validation_0-mlogloss:4.45247
[16]	validation_0-mlogloss:4.44939
[17]	validation_0-mlogloss:4.44665
[18]	validation_0-mlogloss:4.44431
[19]	validation_0-mlogloss:4.44220
[20]	validation_0-mlogloss:4.44027
[21]	validation_0-mlogloss:4.43847
[22]	validation_0-mlogloss:4.43700
[23]	validation_0-mlogloss:4.43564
[24]	validation_0-mlogloss:4.43442
[25]	validation_0-mlogloss:4.43336
[26]	validation_0-mlogloss:4.43250
[27]	validation_0-mlogloss:4.43174
[28]	validation_0-mlogloss:4.4

In [47]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Classification Report:\n", classification_rep)

Accuracy: 0.02
Precision: 0.02
Recall: 0.02
F1 Score: 0.01
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        94
           1       0.02      0.05      0.03        88
           2       0.00      0.00      0.00       110
           3       0.04      0.01      0.01       184
           4       0.01      0.01      0.01        96
           5       0.21      0.04      0.06       191
           6       0.01      0.03      0.02        71
           7       0.00      0.00      0.00       109
           8       0.00      0.00      0.00       298
           9       0.00      0.00      0.00       110
          10       0.00      0.00      0.00        88
          11       0.00      0.00      0.00       104
          12       0.00      0.00      0.00       117
          13       0.00      0.00      0.00        99
          14       0.00      0.00      0.00       100
          15       0.02      0.02      0.02        8

c:\Users\Acer\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Acer\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Acer\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier

In [48]:
# Predict for new resumes (example)
new_resume_similarity = pd.DataFrame({'jaccard_similarity': [0.7, 0.4, 0.9]})  # Example new data
new_resume_pred_proba = model.predict_proba(new_resume_similarity)  # Probability distribution

# Map job_id to job_title (assuming jobreq_df is available)
jobreq_df = pd.read_csv('all_jobreq_extractedentity.csv')
job_id_to_title = dict(zip(jobreq_df.index, jobreq_df['job_title']))

# Format the output
for i, proba in enumerate(new_resume_pred_proba):
    most_suitable_job_id = np.argmax(proba)  # Job ID with the highest probability
    most_suitable_job_title = job_id_to_title.get(most_suitable_job_id, "Unknown Job")
    confidence_percentage = proba[most_suitable_job_id] * 100  # Confidence score as percentage

    print(f"resume{i+1} assessment:")
    print(f"\t-> predicted relevant job: \"{most_suitable_job_title}\"")
    print(f"\t-> percentage: \"{confidence_percentage:.0f}%\"")

resume1 assessment:
	-> predicted relevant job: "FOOD SERVER"
	-> percentage: "19%"
resume2 assessment:
	-> predicted relevant job: "FOOD SERVER"
	-> percentage: "19%"
resume3 assessment:
	-> predicted relevant job: "FOOD SERVER"
	-> percentage: "19%"


In [25]:
# Save the model for future use
model.save_model('C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/models/xgboost_job_matching_multi_class_model.json')